# DuckDB Adzuna store

In [ ]:
#|default_exp utils.adzuna_store

In [ ]:
#|export
import threading

import duckdb
import pyarrow as pa

from ai_index.const import adzuna_db_path

# Canonical schema for the ads table. All string-like columns use VARCHAR.
# year/month are partition columns added during ingest.
_ADS_SCHEMA = [
    ("id", "BIGINT NOT NULL"),
    ("year", "INTEGER NOT NULL"),
    ("month", "INTEGER NOT NULL"),
    ("date_created", "VARCHAR"),
    ("title", "VARCHAR"),
    ("description", "VARCHAR"),
    ("location_raw", "VARCHAR"),
    ("geo_lat", "VARCHAR"),
    ("geo_lng", "VARCHAR"),
    ("LAD22CD", "VARCHAR"),
    ("LAD22NM", "VARCHAR"),
    ("LAD24CD", "VARCHAR"),
    ("category_id", "BIGINT"),
    ("category_name", "VARCHAR"),
    ("company_raw", "VARCHAR"),
    ("company_id", "DOUBLE"),
    ("normalised_company", "VARCHAR"),
    ("salary_min", "DOUBLE"),
    ("salary_max", "DOUBLE"),
    ("salary_currency", "VARCHAR"),
    ("salary_predicted", "DOUBLE"),
    ("salary_raw", "VARCHAR"),
    ("contract_type", "VARCHAR"),
    ("contract_time", "VARCHAR"),
    ("soc2020_major_group", "VARCHAR"),
    ("soc2020_submajor_group", "VARCHAR"),
    ("soc2020_minor_group", "VARCHAR"),
    ("soc2020", "VARCHAR"),
    ("sic_section", "VARCHAR"),
    ("SIC1", "VARCHAR"),
    ("SIC2", "VARCHAR"),
    ("SIC3", "VARCHAR"),
    ("SIC4", "VARCHAR"),
]

# Columns that may be numeric (double) in older parquets and need CAST to VARCHAR
_CAST_IF_NUMERIC = {
    "geo_lat", "geo_lng",
    "soc2020_major_group", "soc2020_submajor_group",
    "soc2020_minor_group", "soc2020",
}


import time
from pathlib import Path


class _ReadWriteLock:
    """Writers-preference readers-writer lock for DuckDB connection serialization.

    Allows multiple concurrent readers (read-only connections) but gives
    exclusive access to writers (read-write connections). Writers take
    priority over waiting readers to prevent writer starvation.
    """

    def __init__(self):
        self._cond = threading.Condition(threading.Lock())
        self._readers = 0
        self._writers_waiting = 0
        self._writer_active = False

    def acquire_read(self):
        with self._cond:
            while self._writer_active or self._writers_waiting > 0:
                self._cond.wait()
            self._readers += 1

    def release_read(self):
        with self._cond:
            self._readers -= 1
            if self._readers == 0:
                self._cond.notify_all()

    def acquire_write(self):
        with self._cond:
            self._writers_waiting += 1
            while self._readers > 0 or self._writer_active:
                self._cond.wait()
            self._writers_waiting -= 1
            self._writer_active = True

    def release_write(self):
        with self._cond:
            self._writer_active = False
            self._cond.notify_all()


class _LockedConnection:
    """DuckDB connection wrapper that releases a read-write lock on close."""

    def __init__(self, conn, release_fn):
        self._conn = conn
        self._release = release_fn
        self._closed = False

    def __getattr__(self, name):
        return getattr(self._conn, name)

    def close(self):
        if not self._closed:
            self._closed = True
            try:
                self._conn.close()
            finally:
                self._release()


_adzuna_rwlock = _ReadWriteLock()


def duckdb_connect_retry(
    db_path: str | Path,
    *,
    read_only: bool = False,
    memory_limit: str | None = None,
    timeout: float = 300,
    poll_interval: float = 2,
) -> duckdb.DuckDBPyConnection:
    """Open a DuckDB connection with retry on lock/configuration conflicts.

    Retries on IOException (lock held by another process) and
    ConnectionException (e.g. read-only vs read-write config mismatch)
    until *timeout* seconds have elapsed.
    """
    deadline = time.monotonic() + timeout
    while True:
        try:
            conn = duckdb.connect(str(db_path), read_only=read_only)
            if memory_limit is not None:
                conn.execute(f"SET memory_limit = '{memory_limit}'")
            return conn
        except (duckdb.IOException, duckdb.ConnectionException) as e:
            if time.monotonic() >= deadline:
                raise
            print(f"duckdb_connect_retry: {db_path} unavailable ({e}), retrying in {poll_interval}s...")
            time.sleep(poll_interval)


def get_adzuna_conn(read_only: bool = False, *, memory_limit: str | None = None, timeout: float = 300, poll_interval: float = 2) -> duckdb.DuckDBPyConnection:
    """Open a DuckDB connection to the Adzuna database.

    Serializes access with a readers-writer lock: multiple read-only connections
    can coexist, but read-write connections get exclusive access. This prevents
    DuckDB's in-process configuration conflict errors when concurrent pipelines
    access the same database.

    The returned connection wrapper releases the lock when close() is called.
    Always close the connection when done.
    """
    if read_only:
        _adzuna_rwlock.acquire_read()
        release_fn = _adzuna_rwlock.release_read
    else:
        _adzuna_rwlock.acquire_write()
        release_fn = _adzuna_rwlock.release_write

    try:
        conn = duckdb_connect_retry(adzuna_db_path, read_only=read_only, memory_limit=memory_limit, timeout=timeout, poll_interval=poll_interval)
    except Exception:
        release_fn()
        raise

    return _LockedConnection(conn, release_fn)


def ensure_ads_table(conn: duckdb.DuckDBPyConnection) -> None:
    """Create the ads table and indexes if they don't exist."""
    cols = ", ".join(f"{name} {dtype}" for name, dtype in _ADS_SCHEMA)
    conn.execute(f"CREATE TABLE IF NOT EXISTS ads ({cols})")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_ads_id ON ads(id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_ads_year_month ON ads(year, month)")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS _meta (
            key VARCHAR PRIMARY KEY,
            value VARCHAR
        )
    """)


def build_insert_from_parquet(
    parquet_path: str,
    year: int,
    month: int,
    source_schema: list[str] | None = None,
) -> str:
    """Build an INSERT INTO ads SELECT ... FROM read_parquet(...) statement.

    Handles schema differences: casts numeric columns to VARCHAR,
    fills missing columns with NULL, and adds year/month.

    Args:
        parquet_path: Path to the parquet file.
        year: Year partition value.
        month: Month partition value.
        source_schema: Column names in the source parquet. If None,
            reads from the file metadata.
    """
    if source_schema is None:
        import pyarrow.parquet as pq
        source_schema_obj = pq.read_schema(parquet_path)
        source_cols = set(source_schema_obj.names)
        # Build a map of source column types for cast decisions
        source_types = {f.name: f.type for f in source_schema_obj}
    else:
        source_cols = set(source_schema)
        source_types = {}

    select_exprs = []
    for col_name, _ in _ADS_SCHEMA:
        if col_name == "year":
            select_exprs.append(f"{year} AS year")
        elif col_name == "month":
            select_exprs.append(f"{month} AS month")
        elif col_name not in source_cols:
            select_exprs.append(f"NULL AS {col_name}")
        elif col_name in _CAST_IF_NUMERIC and source_types:
            # Check if source type is numeric (double/float)
            src_type = source_types.get(col_name)
            if src_type is not None and (pa.types.is_floating(src_type) or pa.types.is_integer(src_type)):
                select_exprs.append(f"CAST({col_name} AS VARCHAR) AS {col_name}")
            else:
                select_exprs.append(col_name)
        else:
            select_exprs.append(col_name)

    select_clause = ", ".join(select_exprs)
    return f"INSERT INTO ads SELECT {select_clause} FROM read_parquet('{parquet_path}')"


def get_ads_by_id(
    ids: list[int],
    columns: list[str] | None = None,
    memory_limit: str | None = None,
) -> pa.Table:
    """Retrieve job ad rows by ID from the Adzuna DuckDB store.

    Args:
        ids: List of job ad IDs to retrieve.
        columns: Columns to include in the result. If None, all columns
            are returned. The ``id`` column is always included.

    Returns:
        A pyarrow Table with matching rows.
    """
    if columns is not None and "id" not in columns:
        columns = ["id"] + list(columns)

    col_clause = ", ".join(columns) if columns else "*"
    placeholders = ", ".join(["?"] * len(ids))

    conn = get_adzuna_conn(read_only=True, memory_limit=memory_limit)
    try:
        result = conn.execute(
            f"SELECT {col_clause} FROM ads WHERE id IN ({placeholders})", ids
        ).fetch_arrow_table()
    finally:
        conn.close()
    return result

def print_ads(*ids: int, width: int = 80) -> None:
    """Pretty-print one or more job ads by ID.

    Usage:
        print_ads(12345)
        print_ads(12345, 67890, 11111)
        print_ads(12345, width=120)
    """
    import textwrap

    table = get_ads_by_id(list(ids), columns=["id", "title", "category_name", "description",
                                               "location_raw", "company_raw", "salary_min", "salary_max"])
    if table.num_rows == 0:
        print(f"No ads found for ids: {ids}")
        return

    for i in range(table.num_rows):
        row = {col: table.column(col)[i].as_py() for col in table.column_names}
        print(f"{'='*width}")
        print(f"ID:       {row['id']}")
        print(textwrap.fill(f"Title:    {row['title']}", width=width, subsequent_indent="          "))
        print(f"Category: {row['category_name']}")
        print(f"Company:  {row['company_raw']}")
        print(f"Location: {row['location_raw']}")
        salary_min, salary_max = row['salary_min'], row['salary_max']
        if salary_min or salary_max:
            print(f"Salary:   {salary_min} - {salary_max}")
        print(f"{'-'*width}")
        print(textwrap.fill(row['description'] or '', width=width))
        print()


def get_all_ad_ids(memory_limit: str | None = None):
    conn = get_adzuna_conn(read_only=True, memory_limit=memory_limit)
    try:
        all_ids = conn.execute("SELECT id FROM ads").fetchnumpy()["id"].tolist()
    finally:
        conn.close()
    return all_ids